# Competitive binding

This tutorial determines the dissociation constants of three unlabeled
competitors by FP competition. Competitor was titrated in 12 three-fold steps,
from 0.56 nM to 100 µM, into 40 nM receptor and 10 nM tracer. The tracer's
dissociation constant, 15 nM, was determined separately by direct binding.
Each competitor was measured in three independent experiments with technical
triplicates. The data are in
{download}`competitive-binding.csv <data/competitive-binding.csv>`.

The x axis is the total competitor concentration in nM.

In [ ]:
import matplotlib.pyplot as plt

import bindcurve as bc

data = bc.DoseResponseData.from_csv("data/competitive-binding.csv", format="wide")
assay = {"RT": 40.0, "LsT": 10.0, "Kds": 15.0}
data.summary()

## Complete competition

`competitor_a` and `competitor_b` displace the tracer completely, so the
three-state model is used, with the assay constants as fixed values.

In [ ]:
complete = bc.fit(data.select(["competitor_a", "competitor_b"]), "comp_3st_specific", fixed=assay)
complete.experiments()[["compound_id", "experiment_id", "Kd", "Kd_stderr", "warnings"]]

In [ ]:
complete.summary()[["compound_id", "N_fit", "Kd", "Kd_CI95_lower", "Kd_CI95_upper", "ymin", "ymax"]]

The fitted `ymin` and `ymax` are the polarization of free and fully bound
tracer. The highest measured polarization is lower than `ymax`, because only
part of the tracer is bound even without competitor.

## Incomplete competition

`competitor_c` lowers the polarization only partly, even at 100 µM: tracer can
still bind the competitor-bound receptor. The four-state model describes this
with `Kd3`, the dissociation constant of tracer from the competitor-bound
receptor. Here `Kd3 = 150 nM` was measured by direct binding of tracer to
receptor saturated with `competitor_c`.

In [ ]:
incomplete = bc.fit(data.select("competitor_c"), "comp_4st_specific", fixed={**assay, "Kd3": 150.0})
incomplete.summary()[["compound_id", "N_fit", "N_flagged", "Kd", "Kd_CI95_lower", "Kd_CI95_upper"]]

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3.5))
bc.plot_compounds(complete, colors=["C0", "C1"], ax=ax)
bc.plot_compounds(incomplete, colors="C2", ax=ax)
ax.set_xlabel("total competitor (nM)")
ax.set_ylabel("polarization (mP)")
ax.legend()
plt.show()

In [ ]:
for results in (complete, incomplete):
    print(results.report(uncertainty="ci95", unit="nM").to_string(index=False))

## Converting an IC₅₀

When only an IC₅₀ is available, `coleska` converts it to a $K_d$ with the same
assay constants. Fitting the logistic model to the complete-competition data
and converting the summary IC₅₀ values:

In [ ]:
ic50 = bc.fit(data.select(["competitor_a", "competitor_b"]), "ic50").summary()
ic50["Kd_coleska"] = bc.coleska(ic50["IC50"], **assay)
ic50["Kd_3st"] = complete.summary()["Kd"]
ic50[["compound_id", "IC50", "Kd_coleska", "Kd_3st"]]

The converted values agree with the $K_d$ from the three-state fit. Under these
assay conditions the IC₅₀ is about four times the $K_d$. The conversion gives no
spread, so the three-state fit is preferable when the assay constants are
known.

The `comp_3st_total` and `comp_4st_total` models are used in the same way, with
the nonspecific-binding factor `N` as an additional fixed value.